# Week 4 – Predictive Modeling and Evaluation
**Project:** Student Placement Prediction and Skill-Gap Analysis Using Python

This notebook removes exact duplicate rows, splits the data with stratification, fits preprocessing only on the training data through Scikit-learn pipelines, and compares a majority baseline, Logistic Regression, Decision Tree and Random Forest.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "raw" / "student_placement_raw_data.csv"
df = pd.read_csv(DATA_FILE)
print("Original shape:", df.shape)
print("Target counts before duplicate removal:\n", df["placement_status"].value_counts(dropna=False))
df = df.drop_duplicates().copy()
print("Shape after exact duplicate removal:", df.shape)
print("Repeated student IDs:", df["student_id"].duplicated().sum())

In [ ]:
target = "placement_status"
features = ["department", "cgpa", "backlogs", "internship_experience", "projects_completed", "certifications", "aptitude_score", "technical_score", "communication_skill"]
X, y = df[features], df[target]
categorical = ["department", "internship_experience", "projects_completed", "certifications", "communication_skill"]
numeric = ["cgpa", "backlogs", "aptitude_score", "technical_score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train target proportions:\n", y_train.value_counts(normalize=True).round(3))
print("Test target proportions:\n", y_test.value_counts(normalize=True).round(3))

In [ ]:
preprocessor = ColumnTransformer([("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric), ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical)])
models = {"Majority-class baseline": DummyClassifier(strategy="most_frequent"), "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42), "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42), "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42)}

In [ ]:
metric_rows, confusion_rows = [], []
for name, estimator in models.items():
    pipe = Pipeline([("preprocessing", preprocessor), ("classifier", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    metric_rows.append({"Model": name, "Accuracy": accuracy_score(y_test, pred), "Precision (Not Placed)": precision_score(y_test, pred, pos_label="Not Placed", zero_division=0), "Recall (Not Placed)": recall_score(y_test, pred, pos_label="Not Placed", zero_division=0), "F1-score (Not Placed)": f1_score(y_test, pred, pos_label="Not Placed", zero_division=0)})
    tp, fn, fp, tn = confusion_matrix(y_test, pred, labels=["Not Placed", "Placed"]).ravel()
    confusion_rows.append({"Model": name, "TP (Not Placed)": tp, "FN (Not Placed)": fn, "FP (Not Placed)": fp, "TN (Placed)": tn})
metrics = pd.DataFrame(metric_rows)
confusions = pd.DataFrame(confusion_rows)
display(metrics.round(3))
display(confusions)

## Interpretation
The minority class is **Not Placed**, so accuracy alone can be misleading. Precision, recall and F1-score in the results table are calculated with **Not Placed** as the positive class. Compare the models using the chosen use case; do not select a model using training accuracy.

In [ ]:
REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)
metrics.to_csv(REPORT_DIR / "week4_model_metrics.csv", index=False)
confusions.to_csv(REPORT_DIR / "week4_confusion_matrices.csv", index=False)
print("Saved metrics and confusion matrices to:", REPORT_DIR)